# Instagram 데이터 중복 정제

같은 `id`가 여러 키워드에 포함된 경우, 지정한 키워드 우선순위가 가장 높은 행 하나만 남깁니다. `like_count`와 `comments_count`의 결측치는 0으로 채워 `int64`로 변환하며, 원본 파일은 수정하지 않고 `instagram_cleaned.csv`로 저장합니다.

In [ ]:
from pathlib import Path
import pandas as pd

# 노트북을 프로젝트 루트 또는 하위 폴더 어디에서 실행해도 CSV를 찾도록 설정
relative_csv_path = Path("02_csv") / "02_instagram" / "instagram_merged.csv"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / relative_csv_path).exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {relative_csv_path}")

input_path = project_root / relative_csv_path
output_path = input_path.with_name("instagram_cleaned.csv")

string_columns = ["id", "caption", "media_type", "permalink", "keyword"]
count_columns = ["like_count", "comments_count"]
dtype_map = {
    **{column: "str" for column in string_columns},
    **{column: "Int64" for column in count_columns},  # 결측치가 있어도 먼저 읽기 위한 임시 자료형
}

df = pd.read_csv(input_path, dtype=dtype_map)
df[string_columns] = df[string_columns].astype("str")

# 일반 int64는 결측치를 저장할 수 없으므로 count 결측치는 0으로 채운 후 변환
missing_counts = df[count_columns].isna().sum()
print("count 컬럼의 0 대체 대상:")
display(missing_counts)
df[count_columns] = df[count_columns].fillna(0).astype("int64")

df["timestamp"] = (
    pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Seoul")
    .astype(pd.DatetimeTZDtype(unit="us", tz="Asia/Seoul"))
)

print(f"전체 행 수: {len(df):,}")
display(df.dtypes)
df.head()

In [ ]:
# id 기준 중복 현황 확인
duplicate_mask = df.duplicated(subset="id", keep=False)
duplicate_rows = df.loc[duplicate_mask].sort_values(["id", "keyword"])

print(f"중복 행 수: {duplicate_mask.sum():,}")
print(f"중복 id 수: {duplicate_rows['id'].nunique(dropna=False):,}")
duplicate_rows[["id", "keyword"]].head(30)

In [ ]:
# 위에 있을수록 우선순위가 높음
keyword_priority = [
    "치이카와",
    "먼작귀",
    "하치와레",
    "우사기",
    "랏코",
    "모몽가",
    "아노코",
    "데카츠요",
    "시사",
    "갑옷씨",
    "쿠리만쥬",
    "카니",
    "파자마 파티즈",
]

# 공백 유무와 관계없이 같은 키워드로 처리 (예: '파자마 파티즈' == '파자마파티즈')
normalize_keyword = lambda value: "".join(str(value).split())
priority_map = {normalize_keyword(keyword): rank for rank, keyword in enumerate(keyword_priority)}
normalized_keyword = df["keyword"].astype("str").str.replace(r"\s+", "", regex=True)
unknown_keywords = sorted(
    df.loc[normalized_keyword.notna() & ~normalized_keyword.isin(priority_map), "keyword"]
    .astype(str)
    .unique()
)
if unknown_keywords:
    print("우선순위에 없는 keyword(가장 낮은 우선순위로 처리):", unknown_keywords)

work = df.assign(
    _keyword_priority=normalized_keyword.map(priority_map).fillna(len(priority_map)),
    _row_order=range(len(df)),
)

# 우선순위가 높은 행을 먼저 배치해 id별 첫 행만 남긴 뒤, 원래 행 순서로 복원
cleaned_df = (
    work.sort_values(["_keyword_priority", "_row_order"], kind="stable")
    .drop_duplicates(subset="id", keep="first")
    .sort_values("_row_order")
    .drop(columns=["_keyword_priority", "_row_order"])
    .reset_index(drop=True)
)

assert not cleaned_df["id"].duplicated().any(), "id 중복이 남아 있습니다."
assert str(cleaned_df["id"].dtype) == "str"
assert str(cleaned_df["timestamp"].dtype) == "datetime64[us, Asia/Seoul]"
assert all(str(cleaned_df[column].dtype) == "int64" for column in count_columns)
cleaned_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"정제 전: {len(df):,}행")
print(f"정제 후: {len(cleaned_df):,}행")
print(f"제거된 중복 행: {len(df) - len(cleaned_df):,}행")
print(f"저장 경로: {output_path}")
display(cleaned_df.dtypes)
cleaned_df.info()
cleaned_df.head()